# Fixed center-cell scenario with left and right neighbors

This notebook verifies a reusable scenario with:

- gNB0 on the left;
- gNB1 in the center and initially serving all UEs;
- gNB2 on the right;
- three stationary eMBB UEs in the center-left overlap;
- three stationary eMBB UEs in the center-right overlap.

The upper action applies release bias only to `(gNB1, eMBB)`. Directional A3 offsets create candidates toward both neighbors, while safe admission limits the executed migration volume.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from global_ppo_3gnb_env import GlobalPPO3GNBEnv
from upper_agent_training_scenarios import CENTER_LEFT_RIGHT_GNB_CONFIGS

SCENARIO = "fixed_center_embb_left_right"
SEED = 123
print(f"Scenario: {SCENARIO}")

In [ ]:
def make_env():
    return GlobalPPO3GNBEnv(
        seed=SEED,
        gnb_configs=CENTER_LEFT_RIGHT_GNB_CONFIGS,
        scenario_mode="curriculum",
        training_scenarios=SCENARIO,
        scenario_selection="cycle",
        upper_window_seconds=1.0,
        local_steps_per_global=10,
        radio_substeps=2,
        terminal_reward_only=False,
        max_handovers_per_local_step=3,
        a3_handover_cooldown_s=2.0,
        a3_min_residence_s=2.0,
    )


def center_embb_action(bias):
    action = np.zeros(9, dtype=np.float32)
    # Flattened [gNB, slice] matrix: gNB1/eMBB is index 3.
    action[3] = float(bias)
    return action


env = make_env()
_obs, reset_info = env.reset(seed=SEED)
ues = list(env.base_env.get_all_ues())
initial_positions = {int(ue.id): (float(ue.x), float(ue.y)) for ue in ues}
initial_loads = np.asarray(reset_info["load_matrix"], dtype=float)[:, 0]
initial_used_prbs = np.array([
    env.base_env.get_slice_used_prbs(gnb_id, "eMBB")
    for gnb_id in range(3)
], dtype=int)
embb_budgets = np.array([
    env.base_env.get_slice_prb_budget(gnb_id, "eMBB")
    for gnb_id in range(3)
], dtype=int)

assert len(ues) == 6
assert all(int(ue.serving_gnb) == 1 for ue in ues)
assert all(float(ue.vx) == 0.0 and float(ue.vy) == 0.0 for ue in ues)
assert sum(float(ue.x) < 0.0 for ue in ues) == 3
assert sum(float(ue.x) > 0.0 for ue in ues) == 3

print("Initial attachments:", {gnb: sum(int(u.serving_gnb) == gnb for u in ues) for gnb in range(3)})
print("Left-overlap UEs:", [int(u.id) for u in ues if u.x < 0])
print("Right-overlap UEs:", [int(u.id) for u in ues if u.x > 0])

## Initial topology

All dots are initially attached to the center gNB even though they lie in left or right overlap areas.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
colors = {0: "#4C78A8", 1: "#F28E2B", 2: "#59A14F"}
names = {0: "Left gNB0", 1: "Center gNB1", 2: "Right gNB2"}

def draw_paper_tower(ax, x, y, color):
    tower_h, half_w = 230.0, 34.0
    ax.plot([x - half_w, x, x + half_w, x - half_w],
            [y, y + tower_h, y, y], color="#34495e", linewidth=1.5)
    for frac in (0.2, 0.4, 0.6, 0.8):
        yy = y + frac * tower_h
        width = half_w * (1.0 - frac)
        ax.plot([x - width, x + width], [yy, yy], color="#34495e", linewidth=0.8)
    ax.scatter([x - 18, x + 18], [y + tower_h - 18, y + tower_h - 35],
               s=[85, 55], color="#666269", zorder=4)
    ax.plot([x, x], [y + tower_h, y + tower_h + 38], color=color, linewidth=2)


for gnb in env.base_env.gnbs:
    gnb_id = int(gnb.id)
    ax.add_patch(Ellipse(
        (float(gnb.x), -15.0),
        width=900.0,
        height=300.0,
        fill=False,
        linestyle=(0, (5, 5)),
        linewidth=1.5,
        color="#3498db",
        alpha=0.85,
    ))
    draw_paper_tower(ax, float(gnb.x), 20.0, colors[gnb_id])
    ax.text(gnb.x, 315, names[gnb_id], ha="center", weight="bold")

for ue in ues:
    ax.scatter(ue.x, ue.y, s=55, color=colors[1], edgecolor="black")
    ax.text(ue.x, ue.y + 18, f"UE{ue.id}", ha="center", fontsize=8)

ax.axvline(0.0, color="gray", linestyle="--", alpha=0.4)
ax.set_aspect("equal")
ax.set_xlim(-1150, 1150)
ax.set_ylim(-210, 390)
ax.set_xlabel("x position (m)")
ax.set_ylabel("y position (m)")
ax.set_title("Paper-style topology: six fixed UEs served by the center cell")
plt.tight_layout()
plt.show()

## Radio condition before applying an offset

All six UEs are covered by all three cells, but their radio margins differ. Four edge-overlap UEs are already strong candidates toward the corresponding outer cell, while the two triple-core UEs see a much stronger center signal and remain protected. This is useful: coverage alone does not force handover.

In [ ]:
radio_rows = []
for ue in ues:
    target_id = 0 if ue.x < 0.0 else 2
    serving = env.base_env._get_gnb_by_id(1)
    target = env.base_env._get_gnb_by_id(target_id)
    rsrp_serving = env.base_env._measure_rsrp(serving, ue)
    rsrp_target = env.base_env._measure_rsrp(target, ue)
    radio_rows.append((
        int(ue.id), target_id, rsrp_serving, rsrp_target,
        rsrp_target - rsrp_serving,
    ))

print(f"{'UE':>3} {'target':>6} {'serving':>10} {'target RSRP':>12} {'delta':>8}")
for ue_id, target_id, serving_rsrp, target_rsrp, delta in radio_rows:
    print(f"{ue_id:3d} {target_id:6d} {serving_rsrp:10.2f} {target_rsrp:12.2f} {delta:8.2f}")

edge_rows = [row for row in radio_rows if abs(initial_positions[row[0]][0]) > 100.0]
core_rows = [row for row in radio_rows if abs(initial_positions[row[0]][0]) <= 100.0]
assert len(edge_rows) == 4 and len(core_rows) == 2
assert all(row[4] > 0.0 for row in edge_rows)
assert all(row[4] < -10.0 for row in core_rows)

## Neutral baseline

With zero center bias, the quota is zero and no UEs should leave.

In [ ]:
neutral_env = make_env()
neutral_env.reset(seed=SEED)
_obs, neutral_reward, _terminated, _truncated, neutral_info = neutral_env.step(
    center_embb_action(0.0)
)
neutral_safe = neutral_info["safe_admission"]

print("Neutral offsets from center:", np.asarray(neutral_info["strong_local_offsets"])[1, :, 0].tolist())
print("Neutral quota:", neutral_safe["quota"].get((1, "eMBB"), 0))
print("Neutral handovers:", neutral_info["handover_count"])

assert neutral_safe["quota"].get((1, "eMBB"), 0) == 0
assert neutral_info["handover_count"] == 0
neutral_env.close()

## Center release action

Bias `-0.8` requests release from center gNB1/eMBB. The lower heuristic applies negative offsets toward both left and right. All eligible UEs are candidates, but safe admission permits only the source quota.

In [ ]:
_obs, release_reward, _terminated, _truncated, release_info = env.step(
    center_embb_action(-0.8)
)
release_safe = release_info["safe_admission"]
events = list(env.base_env.handover_events)

quota = release_safe["quota"][(1, "eMBB")]
used = release_safe["used"][(1, "eMBB")]
center_offsets = np.asarray(release_info["strong_local_offsets"])[1, :, 0]
left_events = [event for event in events if event["to_gnb"] == 0]
right_events = [event for event in events if event["to_gnb"] == 2]

print("Center directional offsets [to left, to right]:", center_offsets.tolist())
print("Window quota:", quota)
print("Used quota:", used)
print("Executed handovers:", release_info["handover_count"])
print("To left:", [event["ue_id"] for event in left_events])
print("To right:", [event["ue_id"] for event in right_events])
print("Remaining at center:", [int(u.id) for u in ues if int(u.serving_gnb) == 1])

assert np.all(center_offsets < 0.0)
assert quota == 4
assert used == 4
assert release_info["handover_count"] == 4
assert len(left_events) > 0 and len(right_events) > 0
assert sum(int(u.serving_gnb) == 1 for u in ues) == 2
assert release_info["handover_count"] <= quota

## Load observation

Slice load is measured from allocated PRBs, not UE count:

$$L_{i,eMBB}=\frac{used\ PRBs_{i,eMBB}}{budget\ PRBs_{i,eMBB}}.$$

The initial center load is `0.90`. Each of the six center UEs contributes about `0.15` load. Bias `-0.8` requests release load `0.48`, which converts to a quota of four UEs. Two move left and two move right, producing balanced loads `[0.30, 0.30, 0.30]`.

In [ ]:
final_loads = np.asarray(release_info["load_matrix"], dtype=float)[:, 0]
final_used_prbs = np.array([
    env.base_env.get_slice_used_prbs(gnb_id, "eMBB")
    for gnb_id in range(3)
], dtype=int)
initial_variance = float(np.var(initial_loads))
final_variance = float(np.var(final_loads))

print(f"{'cell':>8} {'budget':>8} {'used before':>12} {'load before':>12} {'used after':>11} {'load after':>11}")
cell_names = ["left", "center", "right"]
for idx, name in enumerate(cell_names):
    print(
        f"{name:>8} {embb_budgets[idx]:8d} {initial_used_prbs[idx]:12d} "
        f"{initial_loads[idx]:12.2f} {final_used_prbs[idx]:11d} "
        f"{final_loads[idx]:11.2f}"
    )

print("\nRequested release load:", round(release_safe["requested_release_load"][(1, "eMBB")], 3))
print("Estimated load per UE:", round(release_safe["estimated_ue_load"][(1, "eMBB")], 3))
print("Load variance:", round(initial_variance, 4), "->", round(final_variance, 4))
print("Load imbalance:", round(release_info["load_imbalance_start"], 4), "->", round(release_info["load_imbalance_end"], 4))
print("Window reward:", round(float(release_reward), 4))

assert np.allclose(initial_loads, [0.0, 0.9, 0.0])
assert np.allclose(final_loads, [0.3, 0.3, 0.3])
assert initial_used_prbs.sum() == final_used_prbs.sum() == 90
assert final_variance < initial_variance
assert release_info["load_imbalance_end"] < release_info["load_imbalance_start"]
assert release_reward > 0.0

In [ ]:
x = np.arange(3)
width = 0.36
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(x - width / 2, initial_loads, width, label="before", color="#B9CDE5")
axes[0].bar(x + width / 2, final_loads, width, label="after", color="#4C78A8")
axes[0].set_xticks(x, ["left", "center", "right"])
axes[0].set_ylim(0.0, 1.0)
axes[0].set_ylabel("eMBB PRB load")
axes[0].set_title("Per-cell load observation")
axes[0].legend()

axes[1].bar(["before", "after"], [initial_variance, final_variance], color=["#E15759", "#59A14F"])
axes[1].set_ylabel("Variance across cells")
axes[1].set_title("Load imbalance reduction")

plt.tight_layout()
plt.show()

## Final topology after one upper window

UE positions do not change. Only serving-cell attachment changes.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for gnb in env.base_env.gnbs:
    gnb_id = int(gnb.id)
    ax.add_patch(Ellipse(
        (float(gnb.x), -15.0), width=900.0, height=300.0,
        fill=False, linestyle=(0, (5, 5)), linewidth=1.5,
        color="#3498db", alpha=0.85,
    ))
    draw_paper_tower(ax, float(gnb.x), 20.0, colors[gnb_id])
    ax.text(gnb.x, 315, names[gnb_id], ha="center", weight="bold")

for ue in ues:
    serving_id = int(ue.serving_gnb)
    ax.scatter(ue.x, ue.y, s=65, color=colors[serving_id], edgecolor="black")
    ax.text(ue.x, ue.y + 18, f"UE{ue.id}", ha="center", fontsize=8)

ax.set_aspect("equal")
ax.set_xlim(-1150, 1150)
ax.set_ylim(-210, 390)
ax.set_xlabel("x position (m)")
ax.set_ylabel("y position (m)")
ax.set_title("After one window: four admitted, two protected by quota")
plt.tight_layout()
plt.show()

for ue in ues:
    assert np.allclose((ue.x, ue.y), initial_positions[int(ue.id)])
    assert float(ue.vx) == 0.0 and float(ue.vy) == 0.0

In [ ]:
print("PASS: fixed center UEs generated left/right A3 handovers, and safe admission prevented total migration.")
env.close()